# [1장 3강] - 실습: 행렬곱과 완전연결층

**실습 목표**

- 데이터를 (샘플 수 × 특성 수) 행렬로 묶고 각 축의 의미를 설명할 수 있다.
- 행렬곱이 가능한 shape 조건을 설명하고 shape 오류를 진단·수정할 수 있다.
- 완전연결층 `y = XW + b`를 NumPy로 구현하고 결과 shape을 추론할 수 있다.
- 전치행렬의 성질 `(AB)ᵀ = BᵀAᵀ`를 예제로 검증할 수 있다.

**실습에 필요한 데이터셋/파일**

- 분야: 마케팅
- 데이터셋: UCI Bank Marketing
- 사용 방식: `ucimlrepo.fetch_ucirepo(id=222)`
- 출처: https://archive.ics.uci.edu/dataset/222/bank+marketing
- 사용 목적: 고객별 수치형 특성(나이, 잔액, 통화 시간 등)을 데이터 행렬 X로 만들어 완전연결층 연산을 실습합니다.
- 준비물: Python, NumPy, pandas, scikit-learn, ucimlrepo

- 이번 실습은 shape 흐름을 눈으로 확인하는 것이 목적이므로, 전체 데이터가 아니라 무작위 8명 × 5개 특성만 잘라 사용합니다. 작은 행렬이어야 shape 변화를 추적하기 쉽습니다.
- 특성은 age, balance, duration, pdays, previous처럼 값이 실제로 흩어져 있는(분산이 있는) 컬럼만 명시적으로 선택합니다. day_of_week, campaign처럼 앞쪽 행에서 값이 거의 상수인 컬럼을 그대로 쓰면 표준화 후 전부 0이 되어 행렬곱 결과를 해석할 수 없습니다. 같은 이유로 행도 앞 8행이 아니라 무작위 8명을 뽑습니다.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# %pip -q install ucimlrepo scikit-learn pandas numpy

def load_uci(dataset_id):
    """UCI에서 데이터를 불러오고, 실패하면 구조가 비슷한 대체 데이터를 사용합니다."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=dataset_id)
        X, y = ds.data.features.copy(), ds.data.targets.copy()
        if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
            y = y.iloc[:, 0]
        return X, y
    except Exception as e:
        print('[안내] UCI 로드 실패:', e)
        print('[안내] 대체 데이터로 진행합니다. shape 규칙과 해석은 동일합니다.')
        from sklearn.datasets import make_classification
        Xa, ya = make_classification(n_samples=3000, n_features=12,
                                     n_informative=6, random_state=RANDOM_STATE)
        cols = [f'feature_{i}' for i in range(Xa.shape[1])]
        return pd.DataFrame(Xa, columns=cols), pd.Series(ya)


def numeric_frame(X):
    """수치형 컬럼만 남기고 결측값을 중앙값으로 채웁니다."""
    Xn = X.select_dtypes(include='number').copy()
    Xn = Xn.replace([np.inf, -np.inf], np.nan)
    return Xn.fillna(Xn.median(numeric_only=True))


X_raw, y_raw = load_uci(222)   # Bank Marketing
Xn = numeric_frame(X_raw)

FEATURE_COLS = ['age', 'balance', 'duration', 'pdays', 'previous']   # 값이 실제로 흩어져 있는(분산 있는) 컬럼만 명시적으로 선택
FEATURE_COLS = [c for c in FEATURE_COLS if c in Xn.columns] or list(Xn.columns[:5])

X_small = Xn[FEATURE_COLS].sample(n=8, random_state=RANDOM_STATE)   # 앞 8행 대신 무작위 8명 샘플링
X = StandardScaler().fit_transform(X_small)      # 스케일을 맞춰 값 비교를 쉽게
print('데이터 행렬 X:', X.shape)

데이터 행렬 X: (8, 5)


In [8]:
X_small
X

array([[ 0.24917151, -0.51885476,  0.48369821, -0.37796447, -0.37796447],
       [ 0.91362888,  2.33403172, -0.94608091, -0.37796447, -0.37796447],
       [-1.1746657 , -0.55796091,  0.92968436, -0.37796447, -0.37796447],
       [ 0.43901647,  0.59194601,  2.04464972,  2.64575131,  2.64575131],
       [ 1.7679312 , -0.8568436 , -0.4476258 , -0.37796447, -0.37796447],
       [-0.88989826, -0.00302607, -0.32957064, -0.37796447, -0.37796447],
       [-1.26958818, -0.04864991, -0.78867403, -0.37796447, -0.37796447],
       [-0.03559593, -0.94064248, -0.94608091, -0.37796447, -0.37796447]])

## 필수 1 : 고객 수천 명의 반응 점수를 한 번에 계산하기
은행 마케팅팀은 텔레마케팅 대상 고객마다 "가입 가능성 점수"를 계산해 우선순위를 정하려 합니다. 고객을 한 명씩 반복문으로 계산하면 데이터가 커질수록 느려지므로, 고객 전체를 하나의 데이터 행렬로 묶어 행렬곱 한 번으로 처리해야 합니다. 이 계산 구조가 신경망의 완전연결층 y = XW + b와 정확히 같습니다.

### 문제 1-1 : 데이터 행렬 X와 완전연결층 shape 확인하기
1. `X`의 shape을 확인하고, 각 축(행/열)이 무엇을 의미하는지 설명합니다.
2. 출력 점수를 3개 만들기 위한 가중치 행렬 `W`와 편향 벡터 `b`를 난수로 생성합니다. `W`와 `b`의 shape을 **직접 정하고 그 근거를 적습니다.**
3. `Y = X @ W + b`를 계산합니다.
4. `X`, `W`, `b`, `Y`의 shape을 출력하고, `Y`의 shape이 왜 그렇게 결정되는지 설명합니다.

In [17]:
# 1. `X`의 shape을 확인하고, 각 축(행/열)이 무엇을 의미하는지 설명합니다.
# print(X.shape)
X_small
# 8 by 5 행렬이며, 행은 하나의 고객의 데이터이고, 열은 각각의 특성에 대한 값이다.

# 2. 출력 점수를 3개 만들기 위한 가중치 행렬 `W`와 편향 벡터 `b`를 난수로 생성합니다. `W`와 `b`의 shape을 **직접 정하고 그 근거를 적습니다.**
# X(8, 5) @ W(5, 3) + b(3,)= Y(8, 3)
W = np.random.randn(5, 3)
b = np.random.randn(3)
W
b
print(b)

# 3. `Y = X @ W + b`를 계산합니다.
Y = X @ W + b
Y


# 4. `X`, `W`, `b`, `Y`의 shape을 출력하고, `Y`의 shape이 왜 그렇게 결정되는지 설명합니다.
print(f"X: {X.shape}")
print(f"W: {W.shape}")
print(f"b: {b.shape}")
print(f"Y: {Y.shape}")

[0.40405086 1.8861859  0.17457781]
X: (8, 5)
W: (5, 3)
b: (3,)
Y: (8, 3)


### 문제 1-2 : 반복문 계산과 행렬곱 결과가 같은지 확인하기
1. `for` 문으로 고객 한 명씩 `x @ W + b`를 계산해 리스트에 담고 배열로 만듭니다.
2. 문제 1-1에서 계산한 `Y`와 값이 같은지 `np.allclose`로 확인합니다.
3. 두 방식의 결과 shape이 같은지 확인합니다.
4. 배치로 묶어 계산하는 방식의 장점을 한 문장으로 작성합니다.

In [32]:
# 1. `for` 문으로 고객 한 명씩 `x @ W + b`를 계산해 리스트에 담고 배열로 만듭니다.
# X(8, 5) @ W(5, 3) + b(3,)= Y(8, 3)
results = []
for x in X:
    score = x @ W + b
    results.append(score)
    # print(results)

Y_loop = np.array(results)
Y_loop

# 2. 문제 1-1에서 계산한 `Y`와 값이 같은지 `np.allclose`로 확인합니다.
print(np.allclose(Y, Y_loop))

# 3. 두 방식의 결과 shape이 같은지 확인합니다.
print(np.allclose(Y.shape, Y_loop.shape))

# 4. 배치로 묶어 계산하는 방식의 장점을 한 문장으로 작성합니다.

True
True


## 필수 2 : shape 오류를 진단하고 전치로 해결하기
모델 코드를 작성하다 보면 ValueError: matmul: Input operand ... mismatch 같은 오류를 자주 만납니다. 이 오류는 대부분 행렬곱 조건이 맞지 않아 생기며, 축의 의미를 되짚어 보면 대부분 전치로 해결됩니다. 오류를 일부러 만들어 보고 원인과 해결 방법을 익힙니다.

### 문제 2-1 : shape 오류를 재현하고 수정하기
1. 입력 차원과 맞지 않는 가중치 `W_wrong`(예: `(4, 3)`)을 만듭니다.
2. `X.shape`와 `W_wrong.shape`을 나란히 출력하고, `X @ W_wrong`을 시도합니다.
3. `try/except`로 오류 메시지를 출력하고, 앞서 출력한 두 shape과 대조해 어떤 차원이 맞지 않는지 확인합니다. (numpy 버전에 따라 오류 메시지가 전체 shape을 보여주지 않을 수 있으므로, shape 출력과 함께 판단합니다.)
4. 올바른 shape의 `W_fixed`로 수정해 다시 계산합니다.
5. 행렬곱이 가능한 조건을 한 문장으로 정리합니다.

In [38]:
# 1. 입력 차원과 맞지 않는 가중치 `W_wrong`(예: `(4, 3)`)을 만듭니다.
W_wrong = np.random.randn(4, 3)
W_wrong

# 2. `X.shape`와 `W_wrong.shape`을 나란히 출력하고, `X @ W_wrong`을 시도합니다.
# (8, 5) @ (4, 3) + (3,) = 
print(f"X.shape: {X.shape}, \nW_wrong.shape: {W_wrong.shape}")
# Y_wrong = X @ W_wrong + b

# 3. `try/except`로 오류 메시지를 출력하고, 앞서 출력한 두 shape과 대조해 어떤 차원이 맞지 않는지 확인합니다. (numpy 버전에 따라 오류 메시지가 전체 shape을 보여주지 않을 수 있으므로, shape 출력과 함께 판단합니다.)
try:
    Y_wrong = X @ W_wrong + b
except ValueError as e:
    print(e)

# 4. 올바른 shape의 `W_fixed`로 수정해 다시 계산합니다.
# 5. 행렬곱이 가능한 조건을 한 문장으로 정리합니다.

X.shape: (8, 5), 
W_wrong.shape: (4, 3)
matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 5)


### 문제 2-2 : 전치로 shape 맞추고 전치 성질 검증하기
1. `X.T`의 shape을 확인하고 원본과 어떻게 달라졌는지 설명합니다.
2. `(X @ W).T`와 `W.T @ X.T`를 각각 계산해 shape과 값이 같은지 확인합니다.
3. 비교용으로 `X.T @ W.T`가 계산 가능한지 확인하고, 안 된다면 그 이유를 적습니다.
4. 전치를 취할 때 곱하는 순서가 바뀌는 이유를 한 문장으로 작성합니다.

In [ ]:
# 1. `X.T`의 shape을 확인하고 원본과 어떻게 달라졌는지 설명합니다.
print(f"X.shape: {X.shape}, \nX.T.shape: {X.T.shape}")

# 2. `(X @ W).T`와 `W.T @ X.T`를 각각 계산해 shape과 값이 같은지 확인합니다.
test1 = (X @ W).T
test2 = W.T @ X.T
print(test1.shape, test2.shape)

# 3. 비교용으로 `X.T @ W.T`가 계산 가능한지 확인하고, 안 된다면 그 이유를 적습니다.
# test3 = X.T @ W.T
print(X.T.shape, W.T.shape)
# (5, 8) @ (3, 5)로 되는데, 앞의 행과 뒤의 열의 개수가 갖지 않아 연산이 불가능.

# 4. 전치를 취할 때 곱하는 순서가 바뀌는 이유를 한 문장으로 작성합니다.

X.shape: (8, 5), 
X.T.shape: (5, 8)
(3, 8) (3, 8)
(5, 8) (3, 5)


## 심화 1 : 층을 두 개 쌓았을 때의 shape 흐름 추적하기
실제 모델은 완전연결층 하나로 끝나지 않고 여러 층을 이어 붙입니다. 층을 쌓을 때마다 출력 차원이 다음 층의 입력 차원이 되므로, shape 흐름을 정확히 추적하지 못하면 모델을 조립할 수 없습니다. 은닉층을 하나 추가해 데이터가 어떤 shape으로 흘러가는지 확인합니다.

### 문제 3-1 : 2층 구조의 shape 흐름과 파라미터 수 계산하기
1. 입력 차원 5 → 은닉 차원 4 → 출력 차원 2가 되도록 `W1, b1, W2, b2`를 생성합니다.
2. `H = X @ W1 + b1`, `O = H @ W2 + b2`를 계산합니다.
3. `X → H → O`의 shape 변화를 순서대로 출력합니다.
4. 각 층의 파라미터 개수(W의 원소 수 + b의 원소 수)와 전체 파라미터 수를 계산합니다.
5. 은닉 차원을 4에서 16으로 바꾸면 출력 shape과 파라미터 수가 어떻게 변하는지 예측한 뒤 실행해 확인합니다.

In [64]:
# 1. 입력 차원 5 → 은닉 차원 4 → 출력 차원 2가 되도록 `W1, b1, W2, b2`를 생성합니다.
# (8, 5) @ W1(5, 4) + b1(4) = (8, 4)
# (8, 4) @ W2(4, 2) + b2(2) = (8, 2)
W1 = np.random.randn(5, 4)
b1 = np.random.randn(4)
W2 = np.random.randn(4, 2)
b2 = np.random.randn(2)
print(f"W1: {W1}, \nb1: {b1}, \n\nW2: {W2}, \nb2: {b2}")

# 2. `H = X @ W1 + b1`, `O = H @ W2 + b2`를 계산합니다.
H = X @ W1 + b1
O = H @ W2 + b2

# 3. `X → H → O`의 shape 변화를 순서대로 출력합니다.
print(f"X: {X.shape} -> H: {H.shape} -> O: {O.shape}")

# 4. 각 층의 파라미터 개수(W의 원소 수 + b의 원소 수)와 전체 파라미터 수를 계산합니다.
# print(f"입력층: W1:{len(W1)} + b1:{len(b1)}, \n은닉층: W2:{len(W2)} + b2:{len(b2)}")
# print(f"전체 파라미터 수: {len(W1)+len(W2)+len(b1)+len(b2)}")
W1_param = W1.size + b1.size
W2_param = W2.size + b2.size
all_param = W1_param + W2_param
print(all_param)

# 5. 은닉 차원을 4에서 16으로 바꾸면 출력 shape과 파라미터 수가 어떻게 변하는지 예측한 뒤 실행해 확인합니다.
W3 = np.random.randn(5, 16)
b3 = np.random.randn(16)
Y3 = X @ W3 + b3
Y3
print(Y3.shape)

W1: [[ 0.99801011 -2.89625538  2.0883747  -0.13958963]
 [ 1.10818282 -1.03990593  0.61277391 -1.05341556]
 [-0.62376896  1.91403135 -0.1906824   0.21743287]
 [ 0.87006773  0.49568189  0.15041891  0.364961  ]
 [ 2.40341559 -0.0576188   0.20109905  1.0506544 ]], 
b1: [ 1.10552593  1.18703031  0.63873022 -1.14300491], 

W2: [[ 1.63343153 -1.14634539]
 [ 0.30263547 -0.75427585]
 [-0.06413835  0.32876241]
 [ 0.32135722  0.42192075]], 
b2: [1.61371127 0.4535343 ]
X: (8, 5) -> H: (8, 4) -> O: (8, 2)
34
(8, 16)
